### **This code is sourced from Josephine Rosencrone Johnsen @ https://github.com/JosephineRosencrone/BSc_project/blob/main/PSF_test.ipynb. It is stored locally for easy accesibility in the pipeline**

### **Initialisation**

In [ ]:
import numpy as np
from scipy.interpolate import CubicSpline
import matplotlib.pyplot as plt
import cv2

**Values to change depending on user setting**

In [ ]:
# PARAMETERS
f_eye_mm = 17.0                     # focal length of the eye (mm)
f_eye_m  = f_eye_mm / 1000.0

pupil_mm = 6.5                      # Pupil size
pupil_m = pupil_mm / 1000.0

view_distance_mm = 500.0        # TWEAKED: typical PC monitor distance 40cm
pixel_pitch_mm = 0.2331            # TWEAKED: Specifically the AG271QX pixel pitch (2560x1440@144hz), 27inch


### Wavelength to diopter

In [ ]:
# Hand-digitized / approximate Marcos-style LCA values with Swiatczak & Schaeffel's key wavelengths enforced:
lambda_nm = np.array([
    450, 538, 570, 630
])

D_lca = np.array([
    +1.10,  # 450 nm  (from S&S / Marcos)
    +0.24,  # 538 nm  (EXACT value from S&S)
    0.00,   # 570 nm  (reference wavelength)
    -0.23  # 630 nm  (EXACT value from S&S)
])

LCA_Marcos = CubicSpline(lambda_nm, D_lca, bc_type='natural')

def lca_defocus(wavelength_nm: float) -> float:
    """
    Approximate Marcos et al. 1999 LCA curve (450–650 nm),
    constrained to match Swiatczak & Schaeffel's key values:
    450 nm = +1.10 D
    538 nm = +0.24 D
    630 nm = -0.23 D
    """
    return float(LCA_Marcos(wavelength_nm))

print(lca_defocus(538))  # Example usage

# Generate wavelengths
wavelengths = np.arange(450, 651, 1)
lca_values = [lca_defocus(wl) for wl in wavelengths]

# Plot
plt.figure(figsize=(5,5))
plt.plot(wavelengths, lca_values)
plt.xlabel("Wavelength (nm)")
plt.ylabel("LCA (Diopters, 570 nm = 0)")
plt.title("Longitudinal Chromatic Aberration")
plt.grid(True)
plt.show()

In [ ]:
# Input data provided by the user 
lambda_nm = np.array([450, 538, 570, 630])
D_lca = np.array([+1.10, +0.24, 0.00, -0.23])


# Cubic spline built ONLY from these 4 points
spline = CubicSpline(lambda_nm, D_lca, bc_type='natural')

# 3rd-order polynomial from Marcos et al. (converted to diopters & shifted)
def z_marcos(lambda_nm):
    return (20.33
            + 0.08534 * lambda_nm
            - 1.12e-4 * lambda_nm**2
            + 5.93e-8 * lambda_nm**3)

def lca_marcos(lambda_nm, lambda_ref=570):
    z = z_marcos(lambda_nm) / 1000.0
    z_ref = z_marcos(lambda_ref) / 1000.0
    return (1/z) - (1/z_ref)

# Generate wavelengths
wavelengths = np.arange(450, 631, 1)
poly_vals = [lca_marcos(wl) for wl in wavelengths]
spline_vals = [spline(wl) for wl in wavelengths]

# Plot
plt.figure(figsize=(7,7))
plt.plot(wavelengths, poly_vals, label="Marcos 3rd-order polynomial (converted to diopters)", linewidth=2)
plt.plot(wavelengths, spline_vals, label="Cubic spline (450, 538, 570, 630 nm only)", linewidth=2)
plt.scatter(lambda_nm, D_lca, color='black', zorder=5, label="Data points used in spline")

plt.xlabel("Wavelength (nm)")
plt.ylabel("LCA (Diopters)")
plt.title("Comparison: Marcos 3rd-order polynomial vs.\nCubic spline fitted to selected points")
plt.grid(True)
plt.legend()
plt.show()

### Find specific parameters

In [ ]:
#Parameters are defined earlier in the file for ease of swapping out values. The code below is the main part of the PSF generation process, which uses the defined parameters and LCA values to compute the blur and PSF for each color channel.

# STEP 1 — Define wavelengths and natural LCA ΔD (relative to 570 nm)


# LCA values from Marcos et al. used in Swiatczak & Schaeffel (2022)
LCA_nat = {
    "B": {"lambda": 450, "dD": +1.10},   # Blue
    "G": {"lambda": 538, "dD": +0.24},   # Green
    "R": {"lambda": 630, "dD": -0.23},   # Red
}


# Convert dioptric defocus ΔD to axial shift Δf
# Using: |Δf| ≈ f_eye^2 * |ΔD|

def delta_f(deltaD):
    # Axial shift in meters from diopters of defocus
    # |Delta F| = f^2 * |Delta D|
    return (f_eye_m**2) * abs(deltaD)


# Convert Δf into retinal blur radius
# Using similar triangles: r_ret ≈ (A/2) * |Δf| / f_eye

def retinal_blur_radius(deltaD):
    # Retinal blur radius (mm)
    # r_retina = (A/2) (|Delta f|/f)
    df = delta_f(deltaD)                 # meters
    r_ret_m = (pupil_m / 2) * (df / f_eye_m)
    return r_ret_m * 1000                # convert m → mm


# Convert retinal blur to screen blur (physical mm)
# Using projection: d_screen = A * L * |ΔD|

def screen_blur_mm(deltaD):
    # Blur circle diameter on the screen (mm)
    # d_screen = A * L * |Delta D|
    return pupil_mm * (view_distance_mm / 1000) * abs(deltaD)


# Convert screen blur diameter to pixel blur diameter

def blur_pixels(deltaD):
    # Blur diameter in pixels (float).
    # N_px = d_screen / p
    d_screen = screen_blur_mm(deltaD)
    return d_screen / pixel_pitch_mm


# Convert pixel blur diameter into PSF radius

def psf_radius_px(deltaD):
    # PSF radius in pxels (float)
    return blur_pixels(deltaD) / 2.0

# Build the PSF kernel for convolution

def wave_optics_psf(deltaD, wavelength_nm=550, grid_size=256, alpha_defocus=50.0):
    """
    Make a wave-optics PSF for a circular pupil with defocus.
    - deltaD: defocus in diopters (total ΔD for the channel)
    - wavelength_nm: wavelength for phase term (not very critical here)
    - grid_size: pupil sampling grid (NxN)
    - alpha_defocus: scales ΔD to phase strength (tune this)
    """
    # Desired blur diameter in pixels from geometric optics
    d_px = blur_pixels(deltaD)
    if d_px < 1.0:
        # Essentially in focus → 1x1 kernel
        return np.array([[1.0]], dtype=np.float32)

    r_target = d_px / 2.0   # target radius in pixels

    # Normalized pupil coordinates in [-1,1]
    N = grid_size
    coords = np.linspace(-1.0, 1.0, N)
    xx, yy = np.meshgrid(coords, coords)
    rho = np.sqrt(xx**2 + yy**2)

    pupil = (rho <= 1.0).astype(np.complex64)

    # Defocus phase term: Φ(ρ) = k * ρ^2
    # k proportional to ΔD via alpha_defocus
    k = alpha_defocus * deltaD
    phase = np.exp(1j * k * (rho**2))

    field_pupil = pupil * phase

    # Fraunhofer diffraction: PSF ∝ |FFT{field}|^2
    field_image = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(field_pupil)))
    psf = np.abs(field_image)**2

    # Normalize PSF (energy = 1)
    psf /= psf.sum()

    # Rescale PSF so that its effective radius matches r_target

    # Compute radial profile to find an "effective radius" (e.g., 80% energy)
    cy, cx = np.array(psf.shape) // 2
    y, x = np.indices(psf.shape)
    rr = np.sqrt((x - cx)**2 + (y - cy)**2)

    # Flatten & sort by radius
    r_flat = rr.ravel()
    psf_flat = psf.ravel()
    order = np.argsort(r_flat)
    r_sorted = r_flat[order]
    psf_sorted = psf_flat[order]

    cum_energy = np.cumsum(psf_sorted)
    cum_energy /= cum_energy[-1]

    # radius at which we reach 80% encircled energy
    idx_80 = np.searchsorted(cum_energy, 0.8)
    r_eff = r_sorted[idx_80]

    # scaling factor to map r_eff -> r_target
    scale = r_target / r_eff if r_eff > 0 else 1.0

    # Now we want a kernel about 2*r_target in diameter
    ker_radius = int(np.ceil(r_target))
    ker_size = 2*ker_radius + 1

    # Build a resized PSF where spatial sampling is scaled
    # Use interpolation via cv2.resize (we treat the original PSF grid as "space" and compress/expand it)
    psf_rescaled = cv2.resize(psf, None, fx=scale, fy=scale,
                              interpolation=cv2.INTER_LINEAR)

    # After resize, re-normalize
    psf_rescaled = psf_rescaled / psf_rescaled.sum()

    # Crop center to desired kernel size
    h, w = psf_rescaled.shape
    cy2, cx2 = h // 2, w // 2
    y1 = max(0, cy2 - ker_radius)
    y2 = min(h, cy2 + ker_radius + 1)
    x1 = max(0, cx2 - ker_radius)
    x2 = min(w, cx2 + ker_radius + 1)

    kernel = psf_rescaled[y1:y2, x1:x2]

    # If cropping near edge made it smaller, pad to ker_size
    kh, kw = kernel.shape
    if kh != ker_size or kw != ker_size:
        pad_y = ker_size - kh
        pad_x = ker_size - kw
        pad_top = pad_y // 2
        pad_bottom = pad_y - pad_top
        pad_left = pad_x // 2
        pad_right = pad_x - pad_left
        kernel = np.pad(kernel,
                        ((pad_top, pad_bottom), (pad_left, pad_right)),
                        mode="constant", constant_values=0)

    # Final normalize just in case
    kernel = kernel.astype(np.float32)
    kernel /= kernel.sum()

    return kernel

"""
# If we want to use a pillbox PSF instead, the function is here
def pillbox_psf(radius_px):
    #Creates a normalized pillbox (disk) PSF.
    if radius_px <= 0.5:
        # effectively zero blur → identity kernel
        return np.array([[1.0]], dtype=np.float32)

    r = int(np.ceil(radius_px))
    size = 2*r + 1

    y, x = np.ogrid[-r:r+1, -r:r+1]
    mask = (x*x + y*y) <= (radius_px**2)

    kernel = mask.astype(np.float32)
    kernel /= kernel.sum()

    return kernel
"""


# Run example — Compute blur & PSF for each channel

channels = ["B", "G", "R"]

for ch in channels:
    deltaD = LCA_nat[ch]["dD"]

    d_mm = screen_blur_mm(deltaD)
    d_px = blur_pixels(deltaD)
    r_px = psf_radius_px(deltaD)

    print(f"\n{ch} channel ({LCA_nat[ch]['lambda']} nm):")
    print(f"  ΔD = {deltaD} D")
    print(f" Optical blur diameter in screen plane from LCA = {d_mm:.2f} mm = {d_px:.2f} px")
    print(f"  PSF radius = {r_px:.3f} px")

    # psf = pillbox_psf(r_px)
    # print(f"  PSF shape: {psf.shape}")